# Kết nối 2 mạng Nơ-ron (Multi-branch Network) trên dữ liệu thực tế Titanic

Notebook này hướng dẫn từng bước cách xây dựng và kết nối 2 mạng nơ-ron độc lập thông qua **ghép nối (concatenate) 2 tầng ẩn cuối cùng** để giải quyết bài toán phân loại thực tế: Dự đoán khả năng sống sót của hành khách trên tàu **Titanic**.

### Tại sao cần kết nối 2 mạng (Multi-branch Network)?
Trong các bài toán thực tế, dữ liệu đầu vào thường được chia làm nhiều nhóm đặc trưng mang tính chất khác nhau (ví dụ: một nhóm đặc trưng về thông tin cá nhân của người dùng, một nhóm đặc trưng về lịch sử hành vi). Việc tách riêng chúng đi qua các nhánh mạng con (Sub-networks) giúp mỗi nhánh tự tập trung học sâu các đặc trưng chuyên biệt trước khi "hội quân" ở tầng ghép nối cuối cùng để đưa ra dự báo chung. Phương pháp này được gọi là **Late Fusion**.

### Sơ đồ luồng dữ liệu Titanic và Kích thước (Shapes) từng bước:
```
Nhánh A: Đặc trưng nhân khẩu học              Nhánh B: Đặc trưng hành trình & gia đình
(Sex, Age, Pclass)                           (SibSp, Parch, Fare, Embarked_C, Q, S)
     Shape: (N, 3)                                     Shape: (N, 6)
          |
          v                                                 |
      [ Mạng NetA ]                                         v
          |
          v (Tầng ẩn cuối)                              [ Mạng NetB ]
      featA (Shape: (N, 4))                                 |
          \                                                 v (Tầng ẩn cuối)
           \                                            featB (Shape: (N, 4))
            \                                               /
             v                                             v
          [ Ghép nối đặc trưng: torch.cat(dim=1) ]
                            |
                            v
                  feat_combined (Shape: (N, 8))
                            |
                            v
                  [ Tầng phân loại đầu ra (Sigmoid) ]
                            |
                            v
                    y_pred (Shape: (N, 1))
```

* **N**: Kích thước batch (`batch_size`).
* **8** = **4** (đặc trưng cá nhân từ NetA) + **4** (đặc trưng hành trình từ NetB).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Thiết lập random seed
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Thiết bị sử dụng: {device}")

---
## BƯỚC 1: TẢI VÀ TIỀN XỬ LÝ DỮ LIỆU TITANIC VỚI PANDAS

Chúng ta nạp trực tiếp tập dữ liệu Titanic từ URL công khai và xử lý các vấn đề dữ liệu thực tế:
1. Điền giá trị thiếu (NaN) cho cột `Age` và `Fare`.
2. Chuyển đổi thuộc tính phân loại chữ sang số (`Sex` thành 0/1; One-Hot Encoding cho `Embarked`).
3. Phân tách tập dữ liệu thành 2 nhánh đặc trưng độc lập: **Nhánh A (Nhân thân)** và **Nhánh B (Hành trình)**.

In [ ]:
# 1. Tải tập dữ liệu Titanic từ github
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

print(f"Kích thước tập dữ liệu gốc: {df.shape}")
print("Các cột dữ liệu:", df.columns.tolist())

# 2. Tiền xử lý dữ liệu
# Điền giá trị thiếu của Age bằng Median
df['Age'] = df['Age'].fillna(df['Age'].median())
# Điền giá trị thiếu của Fare bằng Median
df['Fare'] = df['Fare'].fillna(df['Fare'].median())
# Điền giá trị thiếu của Embarked bằng giá trị xuất hiện nhiều nhất (Mode)
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# Chuyển giới tính Sex sang số: male -> 0, female -> 1
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

# One-Hot Encoding cho cổng lên tàu Embarked (tạo ra các cột Embarked_C, Embarked_Q, Embarked_S)
df = pd.get_dummies(df, columns=['Embarked'], drop_first=False)

# Chuyển các cột boolean của get_dummies sang dạng float để PyTorch huấn luyện
for col in ['Embarked_C', 'Embarked_Q', 'Embarked_S']:
    df[col] = df[col].astype(float)

# 3. Phân tách cột thành 2 nhánh đầu vào
# Nhánh A (3 đặc trưng): Sex, Age, Pclass
X_A = df[['Sex', 'Age', 'Pclass']].values

# Nhánh B (6 đặc trưng): SibSp (số anh chị em), Parch (số con cái/bố mẹ đi cùng), Fare (giá vé), Embarked_C, Q, S
X_B = df[['SibSp', 'Parch', 'Fare', 'Embarked_C', 'Embarked_Q', 'Embarked_S']].values

# Nhãn Survived
y = df['Survived'].values.reshape(-1, 1)

print(f"Kích thước Nhánh A (X_A): {X_A.shape}")
print(f"Kích thước Nhánh B (X_B): {X_B.shape}")
print(f"Kích thước Nhãn (y):     {y.shape}")

---
## BƯỚC 2: PHÂN CHIA TẬP TRAIN/VAL VÀ CHUẨN HÓA ĐẶC TRƯNG

Chúng ta chia dữ liệu thành tập Huấn luyện (Train) chiếm 80% và tập Kiểm thử (Validation) chiếm 20%. 
Sau đó, chuẩn hóa đặc trưng bằng `StandardScaler` (phép tính toán này rất quan trọng khi huấn luyện MLP để tránh việc các đặc trưng có giá trị lớn như `Fare` lấn át các đặc trưng nhỏ như `Sex`).

In [ ]:
# 1. Chia Train/Validation
X_A_train, X_A_val, X_B_train, X_B_val, y_train, y_val = train_test_split(
    X_A, X_B, y, test_size=0.2, random_state=42
)

# 2. Chuẩn hóa độc lập cho từng nhánh đầu vào
scaler_A = StandardScaler()
X_A_train = scaler_A.fit_transform(X_A_train)
X_A_val = scaler_A.transform(X_A_val)

scaler_B = StandardScaler()
X_B_train = scaler_B.fit_transform(X_B_train)
X_B_val = scaler_B.transform(X_B_val)

print(f"Huấn luyện trên {len(y_train)} mẫu, đánh giá trên {len(y_val)} mẫu.")

---
## BƯỚC 3: KHỞI TẠO PYTORCH DATASET & DATALOADER

Chúng ta đóng gói các mảng NumPy đã tiền xử lý thành `TitanicDataset` và tạo các `DataLoader` để tự động chia batch.

In [ ]:
class TitanicDataset(Dataset):
    def __init__(self, X_A, X_B, y):
        self.X_A = torch.tensor(X_A, dtype=torch.float32)
        self.X_B = torch.tensor(X_B, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        
    def __len__(self):
        return len(self.y)
        
    def __getitem__(self, idx):
        return self.X_A[idx], self.X_B[idx], self.y[idx]

train_dataset = TitanicDataset(X_A_train, X_B_train, y_train)
val_dataset = TitanicDataset(X_A_val, X_B_val, y_val)

batch_size = 32
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=batch_size, shuffle=False)

# Trích thử 1 batch
batch_xA, batch_xB, batch_y = next(iter(train_loader))
print(f"Kích thước xA của một batch: {batch_xA.shape} -> (batch_size, 3 đặc trưng nhân thân)")
print(f"Kích thước xB của một batch: {batch_xB.shape} -> (batch_size, 6 đặc trưng hành trình)")

---
## BƯỚC 4: ĐỊNH NGHĨA KIẾN TRÚC MẠNG HỢP NHẤT

Chúng ta định nghĩa 2 nhánh mạng con độc lập và mạng kết hợp:
* **`NetA` (Nhánh Nhân Thân)**: Nhận đầu vào kích thước `3`, đầu ra tầng ẩn cuối là **`4`**.
* **`NetB` (Nhánh Hành Trình)**: Nhận đầu vào kích thước `6`, đầu ra tầng ẩn cuối là **`4`**.
* **`CombinedNet` (Hợp nhất)**: Ghép nối đầu ra của 2 nhánh ẩn thành vector kích thước **`8`** ($4 + 4$). Sau đó đưa qua tầng phân loại cuối cùng `fc_out` và dùng hàm `Sigmoid` để tính xác suất sống sót.

In [ ]:
# Nhánh A: Đầu vào 3 đặc trưng -> Tầng ẩn cuối 4 đặc trưng
class NetA(nn.Module):
    def __init__(self, input_dim=3, hidden_dim=4):
        super(NetA, self).__init__()
        self.fc1 = nn.Linear(input_dim, 8)
        self.fc2 = nn.Linear(8, hidden_dim)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.relu(self.fc2(out))
        return out

# Nhánh B: Đầu vào 6 đặc trưng -> Tầng ẩn cuối 4 đặc trưng
class NetB(nn.Module): 
    def __init__(self, input_dim=6, hidden_dim=4):
        super(NetB, self).__init__()
        self.fc1 = nn.Linear(input_dim, 12)
        self.fc2 = nn.Linear(12, hidden_dim)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.relu(self.fc2(out))
        return out

# Mạng CombinedNet ghép nối 2 nhánh mạng bằng torch.cat
class CombinedNet(nn.Module):
    def __init__(self, netA, netB, hidden_dimA=4, hidden_dimB=4, output_dim=1):
        super(CombinedNet, self).__init__()
        self.netA = netA
        self.netB = netB
        
        # Tầng đầu vào fc_out = hidden_dimA + hidden_dimB = 8
        self.fc_out = nn.Linear(hidden_dimA + hidden_dimB, output_dim)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, xA, xB):
        featA = self.netA(xA) # Shape: (batch_size, 4)
        featB = self.netB(xB) # Shape: (batch_size, 4)
        
        # Ghép nối theo chiều cột (dim=1)
        # Shape kết quả: (batch_size, 4 + 4) = (batch_size, 8)
        feat_combined = torch.cat((featA, featB), dim=1)
        
        # Phân loại đầu ra
        logits = self.fc_out(feat_combined) # Shape: (batch_size, 1)
        out = self.sigmoid(logits)
        return out

# Khởi tạo các mô hình và cấu trúc
net_A = NetA(input_dim=3, hidden_dim=4)
net_B = NetB(input_dim=6, hidden_dim=4)
model = CombinedNet(net_A, net_B, hidden_dimA=4, hidden_dimB=4, output_dim=1).to(device)

print(model)

# BCELoss và Optimizer Adam
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)

---
## BƯỚC 5: HUẤN LUYỆN VÀ ĐÁNH GIÁ MÔ HÌNH

Chúng ta tiến hành huấn luyện trong 80 epochs để mô hình tối ưu hóa trọng số cho bài toán phân loại nhị phân thực tế.

In [ ]:
epochs = 80
train_loss_history = []
train_acc_history = []
val_loss_history = []
val_acc_history = []

print("=== BẮT ĐẦU HUẤN LUYỆN TÊN TẬP TITANIC ===")
for epoch in range(epochs):
    # --- Giai đoạn Huấn luyện ---
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_xA, batch_xB, batch_y in train_loader:
        batch_xA, batch_xB, batch_y = batch_xA.to(device), batch_xB.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        predictions = model(batch_xA, batch_xB)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * batch_xA.size(0)
        preds_bin = (predictions >= 0.5).float()
        total += batch_y.size(0)
        correct += (preds_bin == batch_y).sum().item()
        
    epoch_train_loss = running_loss / len(train_dataset)
    epoch_train_acc = correct / total
    train_loss_history.append(epoch_train_loss)
    train_acc_history.append(epoch_train_acc)
    
    # --- Giai đoạn Đánh giá (Validation) ---
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for batch_xA, batch_xB, batch_y in val_loader:
            batch_xA, batch_xB, batch_y = batch_xA.to(device), batch_xB.to(device), batch_y.to(device)
            predictions = model(batch_xA, batch_xB)
            loss = criterion(predictions, batch_y)
            
            val_loss += loss.item() * batch_xA.size(0)
            preds_bin = (predictions >= 0.5).float()
            val_total += batch_y.size(0)
            val_correct += (preds_bin == batch_y).sum().item()
            
    epoch_val_loss = val_loss / len(val_dataset)
    epoch_val_acc = val_correct / val_total
    val_loss_history.append(epoch_val_loss)
    val_acc_history.append(epoch_val_acc)
    
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:2d}/{epochs} | Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.2%} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.2%}")

print("=== HUẤN LUYỆN HOÀN TẤT ===")

---
## BƯỚC 6: TRỰC QUAN HÓA KẾT QUẢ

Chúng ta vẽ đồ thị lịch sử huấn luyện của Loss và Accuracy để xem mô hình có bị hiện tượng quá khớp (Overfitting) hay không.

In [ ]:
plt.figure(figsize=(14, 5))

# Vẽ đồ thị Loss
plt.subplot(1, 2, 1)
plt.plot(train_loss_history, color='tab:red', label='Train Loss')
plt.plot(val_loss_history, color='tab:orange', linestyle='--', label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('BCE Loss')
plt.title('Đồ thị Loss trên tập Train và Val')
plt.legend()
plt.grid(True, alpha=0.3)

# Vẽ đồ thị Accuracy
plt.subplot(1, 2, 2)
plt.plot(train_acc_history, color='tab:blue', label='Train Accuracy')
plt.plot(val_acc_history, color='tab:cyan', linestyle='--', label='Val Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Đồ thị Accuracy trên tập Train và Val')
plt.legend()
plt.grid(True, alpha=0.3)

plt.show()